In [1]:
print("salom")

salom


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Model muvaffaqiyatli yuklandi!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7473.00it/s]


Model muvaffaqiyatli yuklandi!


In [5]:
text = "O‘zbekistonning poytaxti Toshkent."

embedding = model.encode(text)

print("Embedding o‘lchami:", embedding.shape)
print("Birinchi 10 ta qiymat:", embedding[:10])

Embedding o‘lchami: (384,)
Birinchi 10 ta qiymat: [-0.04968903  0.3825024   0.04567253  0.04005539 -0.16672297  0.01535981
  0.2141083   0.13448271 -0.16634092  0.12232175]


In [6]:
import json
from pathlib import Path

file_path = Path("plutosoft-uz-chunks.jsonl")

chunks = []

with file_path.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            chunks.append(json.loads(line))

print("Jami chunklar:", len(chunks))
print("\nBirinchi chunk:")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

Jami chunklar: 79

Birinchi chunk:
{
  "id": "company_1_kompaniya_haqida",
  "title": "1. Kompaniya haqida",
  "topic": "company",
  "service": null,
  "source": "plutosoft-uz-malumot.md",
  "language": "uz",
  "content": "## 1. Kompaniya haqida\n\n- **Nomi:** PlutoSoft (yuridik shakli: PlutoSoft LLC)\n- **Sarlavha:** PlutoSoft — Raqamli yechimlar va dasturiy ta'minot ishlab chiqish\n- **Tavsif:** Bizneslar uchun veb-sayt, web ilova, mobil ilova, Telegram bot, AI integratsiya, AI agentlar hamda CRM/ERP tizimlarini ishlab chiqadi.\n- **Shior:** \"Raqamli dunyoda sizning hamkoringiz\". \"Biznesingiz uchun raqamli tizim\". \"Eski va tizimsiz jarayonlarni yagona ekotizimga birlashtiramiz.\"\n- **Slogan (footer):** Biznesingizni raqamlashtirish va yuqori natijalarga olib chiqish uchun zamonaviy veb-yechimlar yaratamiz.\n- **SEO kalit so'zlar:** veb sayt yaratish, telegram bot yaratish, AI integratsiya, CRM ERP ishlab chiqish, dasturiy ta'minot ishlab chiqish, Uzbekistan software company\n\n

In [7]:
# Umumiy statistika
print("Jami chunklar:", len(chunks))
print("Bo'sh content:", sum(
    1 for chunk in chunks
    if not chunk.get("content", "").strip()
))
print("Takroriy ID lar:", len(chunks) - len({
    chunk["id"] for chunk in chunks
}))

# Bir nechta chunk haqida qisqacha ma'lumot
for chunk in chunks[:5]:
    print(
        f"ID: {chunk['id']} | "
        f"Topic: {chunk['topic']} | "
        f"Content uzunligi: {len(chunk['content'])} belgi"
    )

Jami chunklar: 79
Bo'sh content: 0
Takroriy ID lar: 0
ID: company_1_kompaniya_haqida | Topic: company | Content uzunligi: 1355 belgi
ID: navigation_2_sayt_tuzilmasi_url_lar | Topic: navigation | Content uzunligi: 917 belgi
ID: service_3_xizmatlar_6_ta_yonalish | Topic: service | Content uzunligi: 260 belgi
ID: service_3_1_dasturiy_taminotga_ai_integratsiyasi | Topic: service | Content uzunligi: 721 belgi
ID: service_3_2_telegram_bot_ishlab_chiqish | Topic: service | Content uzunligi: 292 belgi


In [8]:
# Chunklarning matnlarini ajratib olamiz
texts = [chunk["content"] for chunk in chunks]

# Embedding yaratamiz
embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)
print("Birinchi embedding:", embeddings[0][:10])

Batches: 100%|██████████| 5/5 [00:01<00:00,  3.78it/s]

Embedding shape: (79, 384)
Birinchi embedding: [-0.08197746 -0.05071641  0.03466323 -0.04685122 -0.02788728 -0.08682586
  0.04697039  0.03239285 -0.02440271  0.01621069]


In [9]:
import numpy as np

print("Shape:", embeddings.shape)
print("Barcha qiymatlar finite:", np.isfinite(embeddings).all())
print("NaN qiymatlar soni:", np.isnan(embeddings).sum())
print("Vektor normasi:", np.linalg.norm(embeddings[0]))

Shape: (79, 384)
Barcha qiymatlar finite: True
NaN qiymatlar soni: 0
Vektor normasi: 1.0


In [10]:
import json
import numpy as np

output_path = "plutosoft-uz-embedded.jsonl"

with open(output_path, "w", encoding="utf-8") as file:
    for chunk, embedding in zip(chunks, embeddings):
        record = {
            **chunk,
            "embedding": embedding.tolist()
        }
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print(f"{len(chunks)} ta embedding saqlandi: {output_path}")

79 ta embedding saqlandi: plutosoft-uz-embedded.jsonl


In [11]:
import chromadb

print("ChromaDB version:", chromadb.__version__)

ChromaDB version: 1.5.9


In [12]:
import chromadb

# Ma'lumotlarni diskda saqlaydigan client
client = chromadb.PersistentClient(
    path="./chroma_db"
)

# PlutoSoft ma'lumotlari uchun collection
collection = client.get_or_create_collection(
    name="plutosoft_uz",
    metadata={"hnsw:space": "cosine"}
)

print("Collection yaratildi:", collection.name)
print("Hozirgi yozuvlar soni:", collection.count())

Collection yaratildi: plutosoft_uz
Hozirgi yozuvlar soni: 0


Ma’lumotlarni ChromaDB'ga yozish

In [13]:
# Chunk ID'lari
ids = [chunk["id"] for chunk in chunks]

# Asl matnlar
documents = [chunk["content"] for chunk in chunks]

# Metadata: ChromaDB metadata qiymatlari oddiy turlar bo'lishi kerak
metadatas = [
    {
        "title": chunk["title"],
        "topic": chunk["topic"],
        "source": chunk["source"],
        "language": chunk["language"],
        "service": chunk["service"] or "",
    }
    for chunk in chunks
]

# Embeddinglarni oddiy Python list'ga o'tkazamiz
vectors = embeddings.tolist()

# ChromaDB'ga saqlash
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=vectors,
)

print("Saqlangan yozuvlar:", collection.count())

Saqlangan yozuvlar: 79


ChromaDB'dan ma’lumotlarni tekshirish

In [14]:
# Bazadagi yozuvlar soni
print("Jami yozuvlar:", collection.count())

# Birinchi 3 ta yozuvni qayta o'qish
result = collection.get(
    limit=3,
    include=["documents", "metadatas", "embeddings"]
)

for i in range(len(result["ids"])):
    print(f"\nID: {result['ids'][i]}")
    print(f"Topic: {result['metadatas'][i]['topic']}")
    print(f"Content: {result['documents'][i][:150]}...")
    print(f"Embedding o'lchami: {len(result['embeddings'][i])}")

Jami yozuvlar: 79

ID: company_1_kompaniya_haqida
Topic: company
Content: ## 1. Kompaniya haqida

- **Nomi:** PlutoSoft (yuridik shakli: PlutoSoft LLC)
- **Sarlavha:** PlutoSoft — Raqamli yechimlar va dasturiy ta'minot ishla...
Embedding o'lchami: 384

ID: navigation_2_sayt_tuzilmasi_url_lar
Topic: navigation
Content: ## 2. Sayt tuzilmasi (URL lar)

- Bosh sahifa: https://plutosoft.uz
- Xizmatlar: https://www.plutosoft.uz/services
- Portfolio: https://www.plutosoft....
Embedding o'lchami: 384

ID: service_3_xizmatlar_6_ta_yonalish
Topic: service
Content: ## 3. Xizmatlar (6 ta yo'nalish)

Xizmatlar sahifasi: "Biz biznesingiz uchun veb-sayt, web ilova, mobil ilova, Telegram bot, AI integratsiya va maxsus...
Embedding o'lchami: 384
